In [1]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/msawant/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True


1. we join [A] original hasoc dataset including annotations for all tasks (Task1, Task2, Task3) with [B] relabelled dataset <br>

2. Relabelled is only done for original Task1 (HOF, NOT) <br>

3. The tweets were assigned new relabeling by matching tweet IDs from the original dataset with the relabeled dataset [Iteration1] and storing the results in a new column, New_label.<br/>

4. The New_label column stores relabeled annotations from iteration 1. For tweets that were not relabeled in this iteration, the values from Original_label are copied into New_label. <br/>

5. New column "Mismatch" assigned the value of 1 if the annotations between columns [Orginal Label] from Task1 and [New_label] from Iteration1 was different




In [2]:
import pandas as pd
import numpy as np

Original Dataset

In [3]:
file_path="dataset/Hindi_Hasoc2019.tsv"
train_df = pd.read_csv(file_path, sep="\t", encoding="utf-8")
train_df= train_df.dropna(how='all')




In [4]:
train_df.head(20)

,text_id,text,task_1,task_2,task_3
0,hasoc_hi_5556,"बांग्लादेश की शानदार वापसी, भारत को 314 रन पर ...",NOT,NONE,NONE
1,hasoc_hi_5648,सब रंडी नाच देखने मे व्यस्त जैसे ही कोई #शांती...,HOF,PRFN,UNT
2,hasoc_hi_164,तुम जैसे हरामियों के लिए बस जूतों की कमी है शु...,HOF,PRFN,TIN
3,hasoc_hi_3530,"बीजेपी MLA आकाश विजयवर्गीय जेल से रिहा, जमानत ...",NOT,NONE,NONE
4,hasoc_hi_5206,चमकी बुखार: विधानसभा परिसर में आरजेडी का प्रदर...,NOT,NONE,NONE
5,hasoc_hi_5121,मुंबई में बारिश से लोगों को काफी समस्या हो रही है,NOT,NONE,NONE
6,hasoc_hi_7142,Ahmed's dad:-- beta aaj teri mammy kyu nahi ba...,NOT,NONE,NONE
7,hasoc_hi_4321,"5 लाख मुसलमान उर्स में, अजमेर की दरगाह पर आते ...",NOT,NONE,NONE
8,hasoc_hi_4674,"Do mahashaktiyan mili hain, charo taraf khusi ...",NOT,NONE,NONE
9,hasoc_hi_1637,Chants of 'Jai Sri Ram' as Owaisi takes oath: ...,NOT,NONE,NONE


In [5]:
train_df = train_df.rename(columns={'text': 'tweet','task_1': 'Original_label' })


In [6]:
train_df.head(30)

,text_id,tweet,Original_label,task_2,task_3
0,hasoc_hi_5556,"बांग्लादेश की शानदार वापसी, भारत को 314 रन पर ...",NOT,NONE,NONE
1,hasoc_hi_5648,सब रंडी नाच देखने मे व्यस्त जैसे ही कोई #शांती...,HOF,PRFN,UNT
2,hasoc_hi_164,तुम जैसे हरामियों के लिए बस जूतों की कमी है शु...,HOF,PRFN,TIN
3,hasoc_hi_3530,"बीजेपी MLA आकाश विजयवर्गीय जेल से रिहा, जमानत ...",NOT,NONE,NONE
4,hasoc_hi_5206,चमकी बुखार: विधानसभा परिसर में आरजेडी का प्रदर...,NOT,NONE,NONE
5,hasoc_hi_5121,मुंबई में बारिश से लोगों को काफी समस्या हो रही है,NOT,NONE,NONE
6,hasoc_hi_7142,Ahmed's dad:-- beta aaj teri mammy kyu nahi ba...,NOT,NONE,NONE
7,hasoc_hi_4321,"5 लाख मुसलमान उर्स में, अजमेर की दरगाह पर आते ...",NOT,NONE,NONE
8,hasoc_hi_4674,"Do mahashaktiyan mili hain, charo taraf khusi ...",NOT,NONE,NONE
9,hasoc_hi_1637,Chants of 'Jai Sri Ram' as Owaisi takes oath: ...,NOT,NONE,NONE


In [7]:
train_df["Original_label"].value_counts()

Original_label
HOF    2469
NOT    2196
Name: count, dtype: int64

New Dataset

In [ ]:
conda update openpyxl

In [9]:
#incorrect_data=pd.read_excel("dataset/InCorrect_Labels2.xlsx")
incorrect_data=pd.read_excel("dataset/Relabelled_Tweets.xlsx")
incorrect_data=incorrect_data[['Tweet ID','Tweet','Label Change']]
incorrect_data.tail(30)

,Tweet ID,Tweet,Label Change
301,1167,बीजेपी नेता कैलाश विजयवर्गीय पत्रकार से कहता ह...,NOT
302,1166,"राम....जीवन है, राम.....करुणा है,राम.....मर्या...",NOT
303,1163,#नीच कहने पर मणिशंकर को निष्कासित कर दिया गया ...,NOT
304,1140,मीडिया इतना डरी हुई और लाचार क्यो नजर आ रही है...,NOT
305,1139,वंदेमातरम पर राजनीति करने वाले अगर बिना देखे '...,NOT
306,1037,जो लोग गौडसे की पूजा करते हैं वो बाबा रामरहिम ...,NOT
307,1020,3 दिन से फंसे 2 साल के मासूम को अभी तक नही बचा...,NOT
308,878,Vadnde matram ko gane me kitnw time lgta hai y...,NOT
309,876,अगर आपको ये देखना है कि अपनी “फ़ील्ड” का “नाका...,NOT
310,872,#प्रज्ञा_ठाकुर ने #गोडसे_ज़िंदाबाद कहा तो नारा...,NOT


In [10]:
print("Total Length of Relabelled Tweets and Total length of unique relabelled tweets")
unique_relabelled_tweets=len(incorrect_data), incorrect_data["Tweet ID"].nunique()
print(unique_relabelled_tweets)

Total Length of Relabelled Tweets and Total length of unique relabelled tweets
(331, 325)


In [11]:
dups = incorrect_data[incorrect_data["Tweet ID"].duplicated(keep=False)].sort_values("Tweet ID")
print("length of duplicate relabelled tweets are",len(dups))
print("Duplicate Tweets are as follows")
dups

length of duplicate relabelled tweets are 12
Duplicate Tweets are as follows


,Tweet ID,Tweet,Label Change
2,2496,कितना easy सवाल भारत क्या पूरे विश्व को पता है...,HOF
251,2496,कितना easy सवाल भारत क्या पूरे विश्व को पता है...,HOF
6,2642,लीजिये मुजराबाजी शुरू हलाला वालीयों का चोर रंड...,HOF
244,2642,लीजिये मुजराबाजी शुरू हलाला वालीयों का चोर रं...,HOF
239,2679,अबु आझमी तर भड़वा आहे पण हे माझा वाले पन निर्ल...,HOF
329,2679,अबु आझमी तर भड़वा आहे पण हे माझा वाले पन निर्ल...,HOF
7,3808,देशभक्त मोहम्मद शमी ने तीसरे मैच में भी 5 लिया...,HOF
130,3808,देशभक्त मोहम्मद शमी ने तीसरे मैच में भी 5 लिया...,HOF
106,3992,.@ArvindKejriwal केजरीवाल सरकार की तर्ज पर झार...,NOT
134,3992,मोदी जी का मक़सद है जहां चुनाव हो या होने वाला...,NOT


In [12]:
#incorrect_data['Tweet ID'] = incorrect_data['Tweet ID'].astype(int)

#incorrect_data.loc[:, "Tweet ID"] = incorrect_data["Tweet ID"].apply(lambda x: x-2)
#ids_to_check = incorrect_data["Tweet ID"].values.tolist()
#print(ids_to_check)

Creating dictionary from the relabelled dataset

In [13]:
label_map = dict(zip(incorrect_data["Tweet ID"], incorrect_data["Label Change"]))

In [14]:
print("Length of the relabelled tweets info stored in dictionary",len(label_map))
print("Values of the relabelled tweets info stored in dictonary are",label_map)

Length of the relabelled tweets info stored in dictionary 325
Values of the relabelled tweets info stored in dictonary are {3505: 'HOF', 4193: 'HOF', 2496: 'HOF', 1064: 'HOF', 2701: 'HOF', 2558: 'HOF', 2642: 'HOF', 3808: 'HOF', 4655: 'NOT', 4649: 'NOT', 4646: 'NOT', 4645: 'HOF', 4643: 'NOT', 4627: 'NOT', 4622: 'NOT', 4618: 'NOT', 4613: 'NOT', 4612: 'NOT', 4609: 'NOT', 4608: 'NOT', 4602: 'NOT', 4598: 'NOT', 4590: 'NOT', 4588: 'NOT', 4579: 'NOT', 4577: 'NOT', 4576: 'NOT', 4573: 'NOT', 4567: 'NOT', 4566: 'NOT', 4560: 'NOT', 4546: 'NOT', 4544: 'NOT', 4538: 'NOT', 4534: 'NOT', 4533: 'NOT', 4532: 'NOT', 4529: 'NOT', 4528: 'NOT', 4522: 'NOT', 4515: 'NOT', 4511: 'NOT', 4501: 'NOT', 4496: 'NOT', 4487: 'NOT', 4475: 'NOT', 4472: 'NOT', 4464: 'NOT', 4457: 'NOT', 4446: 'NOT', 4433: 'NOT', 4425: 'NOT', 4405: 'NOT', 4403: 'NOT', 4375: 'NOT', 4369: 'HOF', 4325: 'NOT', 4343: 'NOT', 4330: 'NOT', 4328: 'NOT', 4322: 'NOT', 4317: 'NOT', 4310: 'NOT', 4305: 'NOT', 4303: 'NOT', 4294: 'NOT', 4288: 'NOT', 4285:

Sorting the values in dictionary

In [15]:
label_map_sorted = dict(sorted(label_map.items()))
print("IDs of the relabeled tweets stored in the dictionary in ascending order \n",list(label_map_sorted.keys()))

IDs of the relabeled tweets stored in the dictionary in ascending order 
 [13, 101, 103, 123, 273, 407, 471, 472, 474, 540, 594, 597, 633, 660, 814, 871, 872, 876, 878, 1020, 1037, 1064, 1139, 1140, 1163, 1166, 1167, 1179, 1184, 1189, 1210, 1444, 1451, 1513, 1565, 1583, 1802, 1839, 1851, 1871, 1918, 1921, 1937, 1938, 1939, 1951, 1992, 2010, 2069, 2083, 2086, 2088, 2092, 2101, 2107, 2121, 2134, 2144, 2154, 2178, 2240, 2257, 2258, 2259, 2277, 2287, 2311, 2319, 2332, 2344, 2352, 2385, 2420, 2452, 2481, 2489, 2492, 2496, 2529, 2558, 2565, 2571, 2586, 2601, 2635, 2642, 2647, 2662, 2675, 2678, 2679, 2701, 2704, 2707, 2708, 2723, 2728, 2732, 2742, 2743, 2753, 2754, 2756, 2764, 2780, 2796, 2812, 2816, 2826, 2828, 2829, 2832, 2836, 2840, 2845, 2931, 2970, 2971, 2991, 3005, 3008, 3012, 3022, 3023, 3025, 3041, 3043, 3054, 3058, 3067, 3069, 3073, 3079, 3084, 3085, 3090, 3104, 3116, 3121, 3133, 3143, 3159, 3189, 3200, 3209, 3210, 3212, 3216, 3220, 3227, 3229, 3274, 3279, 3331, 3341, 3352, 3354, 335

Assigning the new label to the train dataset

In [16]:
train_df["New_label"] = train_df.index.map(lambda idx: label_map.get(idx + 2, None))


In [17]:
train_df.head(30)

,text_id,tweet,Original_label,task_2,task_3,New_label
0,hasoc_hi_5556,"बांग्लादेश की शानदार वापसी, भारत को 314 रन पर ...",NOT,NONE,NONE,None
1,hasoc_hi_5648,सब रंडी नाच देखने मे व्यस्त जैसे ही कोई #शांती...,HOF,PRFN,UNT,None
2,hasoc_hi_164,तुम जैसे हरामियों के लिए बस जूतों की कमी है शु...,HOF,PRFN,TIN,None
3,hasoc_hi_3530,"बीजेपी MLA आकाश विजयवर्गीय जेल से रिहा, जमानत ...",NOT,NONE,NONE,None
4,hasoc_hi_5206,चमकी बुखार: विधानसभा परिसर में आरजेडी का प्रदर...,NOT,NONE,NONE,None
5,hasoc_hi_5121,मुंबई में बारिश से लोगों को काफी समस्या हो रही है,NOT,NONE,NONE,None
6,hasoc_hi_7142,Ahmed's dad:-- beta aaj teri mammy kyu nahi ba...,NOT,NONE,NONE,None
7,hasoc_hi_4321,"5 लाख मुसलमान उर्स में, अजमेर की दरगाह पर आते ...",NOT,NONE,NONE,None
8,hasoc_hi_4674,"Do mahashaktiyan mili hain, charo taraf khusi ...",NOT,NONE,NONE,None
9,hasoc_hi_1637,Chants of 'Jai Sri Ram' as Owaisi takes oath: ...,NOT,NONE,NONE,None


Calculating HOF/NOT count

In [18]:
train_df[train_df["New_label"]=="HOF"]

,text_id,tweet,Original_label,task_2,task_3,New_label
11,hasoc_hi_288,गाये के नाम पर जुल्म करने वालों तुम्हारी औकात ...,NOT,NONE,NONE,HOF
99,hasoc_hi_495,मुस्लिम लोगों से वोट मांगने वाली पार्टियां क्य...,NOT,NONE,NONE,HOF
101,hasoc_hi_5808,बुर्का भी केवल लाचार मुस्लिम औरतों के लिए है। ...,NOT,NONE,NONE,HOF
271,hasoc_hi_767,अपने विष्लेसण की पुड़िया बना लो और इसको अपने *...,NOT,NONE,NONE,HOF
538,hasoc_hi_5778,चाट ते रहो चाट ते रहो चाट ते रहो चाट ते रहोच...,NOT,NONE,NONE,HOF
592,hasoc_hi_1975,शर्म करो डॉक्टरो एक थप्पड़ खाई और हड़ताल पर...,NOT,NONE,NONE,HOF
595,hasoc_hi_2285,"एक ही शब्द बना है इनके लिए #नीच, सोच से भी और ...",NOT,NONE,NONE,HOF
631,hasoc_hi_4401,बाई कडून पैसा घेणाऱ्याला शुद्ध मराठीत भडवा म्ह...,NOT,NONE,NONE,HOF
812,hasoc_hi_4620,क्या आपने कभी भारत के मुस्लिमो को पाकिस्तान का...,NOT,NONE,NONE,HOF
1062,hasoc_hi_844,"प्राकृतिक वर्फ भरे पहाड़, ठंड भरी ओस से गीली स...",NOT,NONE,NONE,HOF


In [19]:
train_df[train_df["New_label"]=="NOT"]

,text_id,tweet,Original_label,task_2,task_3,New_label
121,hasoc_hi_4608,एसे #नीच और #घटिया लोगो को #चौराहे पर सबके स...,HOF,OFFN,TIN,NOT
405,hasoc_hi_5113,किसी राष्ट्र के प्रधानमंत्री के लिए अपशब्दों ग...,HOF,HATE,TIN,NOT
469,hasoc_hi_7232,#PMModi ने 25 जून को संसद में कहा कि #WaterCri...,HOF,HATE,TIN,NOT
470,hasoc_hi_404,"#AIMIM Chief @asadowaisi says to the Chair, '...",HOF,HATE,TIN,NOT
472,hasoc_hi_664,हनीप्रीत बिना पापा #RamRahim को खेतो में रोटी ...,HOF,HATE,TIN,NOT
...,...,...,...,...,...,...
4625,hasoc_hi_102,#Bihar के लोकप्रिय मुख्यमंत्री आदरणीय श्री @Ni...,HOF,HATE,TIN,NOT
4641,hasoc_hi_338,नीति आयोग की रिपोर्ट सरकार को लज्जित करने वाली...,HOF,HATE,TIN,NOT
4644,hasoc_hi_7422,देश का तिरंगा नीचे । भगवा झंडा तिरंगे से ऊपर ...,HOF,HATE,TIN,NOT
4647,hasoc_hi_4117,तू हवा के रुख पर चाहतो के दीप जलाने कि जिद न क...,HOF,HATE,TIN,NOT


In [20]:
hof_not_count=train_df["New_label"].value_counts()

In [21]:
hof_not_count

New_label
NOT    271
HOF     54
Name: count, dtype: int64

Filling the empty value in New_label column

In [22]:
train_df["New_label"] = train_df["New_label"].fillna(train_df["Original_label"])
#The New_label column stores relabeled annotations from iteration 1. For tweets that were not relabeled in this iteration, the values from Original_label are copied into New_label. <br/>


In [23]:
train_df.head(30)

,text_id,tweet,Original_label,task_2,task_3,New_label
0,hasoc_hi_5556,"बांग्लादेश की शानदार वापसी, भारत को 314 रन पर ...",NOT,NONE,NONE,NOT
1,hasoc_hi_5648,सब रंडी नाच देखने मे व्यस्त जैसे ही कोई #शांती...,HOF,PRFN,UNT,HOF
2,hasoc_hi_164,तुम जैसे हरामियों के लिए बस जूतों की कमी है शु...,HOF,PRFN,TIN,HOF
3,hasoc_hi_3530,"बीजेपी MLA आकाश विजयवर्गीय जेल से रिहा, जमानत ...",NOT,NONE,NONE,NOT
4,hasoc_hi_5206,चमकी बुखार: विधानसभा परिसर में आरजेडी का प्रदर...,NOT,NONE,NONE,NOT
5,hasoc_hi_5121,मुंबई में बारिश से लोगों को काफी समस्या हो रही है,NOT,NONE,NONE,NOT
6,hasoc_hi_7142,Ahmed's dad:-- beta aaj teri mammy kyu nahi ba...,NOT,NONE,NONE,NOT
7,hasoc_hi_4321,"5 लाख मुसलमान उर्स में, अजमेर की दरगाह पर आते ...",NOT,NONE,NONE,NOT
8,hasoc_hi_4674,"Do mahashaktiyan mili hain, charo taraf khusi ...",NOT,NONE,NONE,NOT
9,hasoc_hi_1637,Chants of 'Jai Sri Ram' as Owaisi takes oath: ...,NOT,NONE,NONE,NOT


Finding the Disagreement Tweets

In [24]:
train_df['Mismatch'] = (train_df['Original_label'] != train_df['New_label']).astype(int)

In [25]:
mismatch_count = (train_df["Mismatch"] == 1).sum()
print("Count of tweets with mismatch column having value 1 indicating difference in original and new label",mismatch_count)

Count of tweets with mismatch column having value 1 indicating difference in original and new label 325


In [26]:
train_df.head(20)

,text_id,tweet,Original_label,task_2,task_3,New_label,Mismatch
0,hasoc_hi_5556,"बांग्लादेश की शानदार वापसी, भारत को 314 रन पर ...",NOT,NONE,NONE,NOT,0
1,hasoc_hi_5648,सब रंडी नाच देखने मे व्यस्त जैसे ही कोई #शांती...,HOF,PRFN,UNT,HOF,0
2,hasoc_hi_164,तुम जैसे हरामियों के लिए बस जूतों की कमी है शु...,HOF,PRFN,TIN,HOF,0
3,hasoc_hi_3530,"बीजेपी MLA आकाश विजयवर्गीय जेल से रिहा, जमानत ...",NOT,NONE,NONE,NOT,0
4,hasoc_hi_5206,चमकी बुखार: विधानसभा परिसर में आरजेडी का प्रदर...,NOT,NONE,NONE,NOT,0
5,hasoc_hi_5121,मुंबई में बारिश से लोगों को काफी समस्या हो रही है,NOT,NONE,NONE,NOT,0
6,hasoc_hi_7142,Ahmed's dad:-- beta aaj teri mammy kyu nahi ba...,NOT,NONE,NONE,NOT,0
7,hasoc_hi_4321,"5 लाख मुसलमान उर्स में, अजमेर की दरगाह पर आते ...",NOT,NONE,NONE,NOT,0
8,hasoc_hi_4674,"Do mahashaktiyan mili hain, charo taraf khusi ...",NOT,NONE,NONE,NOT,0
9,hasoc_hi_1637,Chants of 'Jai Sri Ram' as Owaisi takes oath: ...,NOT,NONE,NONE,NOT,0


In [27]:
train_df.to_csv("dataset/Hindi_alltasks.csv",index=False)

In this code, <br/>
1 we join original hasoc dataset with annotations for all tasks (Task1, Task2, Task3) with relabelled dataset <br/>

2 Relabelled is only done for original Task1 <br/>

3 Find the disagreeging tweets between Task1 (Original_Label column) and Relabelled (New_Label column) to produce new column Mismatch